In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Set RETRAIN_BASE = True when you trained locally and uploaded model.pkl +
# value_fn files — Kaggle will skip Phases 1-4 and go straight to bot fine-tuning.
RETRAIN_BASE = True
RUN_VS_BOTS  = False   # set False to skip live-game fine-tuning

WORK_DIR = "/kaggle/working/OrbitsWar"

# ── Auto-detect dataset mount path ────────────────────────────────────────────
import os

def find_dataset_path():
    base = "/kaggle/input"
    if not os.path.exists(base):
        return None
    for root, dirs, files in os.walk(base):
        dirs.sort()
        if "main.py" in files and "records" in dirs:
            print(f"Found OrbitsWar dataset at: {root}")
            return root
    print("Did not find main.py + records/ anywhere under /kaggle/input/")
    print("Full tree:")
    for root, dirs, files in os.walk(base):
        depth = root.replace(base, "").count(os.sep)
        if depth > 4:
            continue
        indent = "  " * depth
        print(f"{indent}{os.path.basename(root)}/  files={files[:5]}")
    return None

INPUT_DIR = find_dataset_path()

if INPUT_DIR is None:
    raise RuntimeError(
        "Could not find OrbitsWar dataset. See tree printout above.\n"
        "Make sure you attached the dataset to this notebook (Add data -> Your datasets)."
    )

print(f"\nInput:   {INPUT_DIR}")
print(f"WorkDir: {WORK_DIR}")
print(f"Retrain base: {RETRAIN_BASE}  |  Vs-bots: {RUN_VS_BOTS}")

In [ ]:
# ── Copy code files to writable working directory and symlink records ─────────
import os, shutil, sys

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

# Copy only python code folders/files and symlink records/
for item in os.listdir(INPUT_DIR):
    item_path = os.path.join(INPUT_DIR, item)
    dest_path = os.path.join(WORK_DIR, item)
    if item == "records":
        os.symlink(item_path, dest_path)
        print(f"Symlinked records/ -> {item_path}")
    elif item in ["classifier", "regressor", "rl", "main.py", "eval_full.py"]:
        if os.path.isdir(item_path):
            shutil.copytree(item_path, dest_path)
        else:
            shutil.copy2(item_path, dest_path)
        print(f"Copied {item}")

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
print(f"\nSetup complete in {WORK_DIR}")

In [ ]:
!pip install -q kaggle-environments scikit-learn joblib
print("Dependencies ready.")

In [ ]:
# ── Verify replay files ────────────────────────────────────────────────────────
import glob
records = glob.glob("records/*.json")
print(f"Replay files: {len(records)}")
if not records:
    print("WARNING: No replay JSON files found in records/")

enemies = glob.glob("enemy agents/*/main.py") + glob.glob("enemy agents/my/*.py")
enemies = [e for e in enemies if not e.endswith("__init__.py")]
print(f"Enemy agents: {len(enemies)}")
for e in sorted(enemies):
    print(f"  {e}")

---
## Phase 1 — 1v1 Classifier (GBC)
Binary win/loss labels. Trains GLOBAL / EARLY / MID / LATE subsets.  
Output: `classifier/value_fn_1v1.py`

In [ ]:
import subprocess
subprocess.run(["python", "classifier/train_from_replays.py"], check=True)

In [ ]:
p = "classifier/value_fn_1v1.py"
print(f"[OK] {p}  ({os.path.getsize(p)/1024:.1f} KB)" if os.path.exists(p) else f"[FAIL] {p}")

---
## Phase 3 — 1v1 Regressor (GBR)
TD-discounted returns: V≈0.97 when dominating at turn 480, V≈0.50 when neutral at turn 100.  
Output: `regressor/value_fn_1v1.py`

In [ ]:
import subprocess
subprocess.run(["python", "regressor/train_from_replays.py"], check=True)

In [ ]:
p = "regressor/value_fn_1v1.py"
print(f"[OK] {p}  ({os.path.getsize(p)/1024:.1f} KB)" if os.path.exists(p) else f"[FAIL] {p}")

In [ ]:
# ── Verify 1v1 models are present ─────────────────
import os
required = [
    ("classifier/value_fn_1v1.py",   "classifier/model.pkl"),
    ("regressor/value_fn_1v1.py",    "regressor/model.pkl"),
]
all_present = True
for vfn, pkl in required:
    vfn_ok = os.path.exists(vfn)
    pkl_ok = os.path.exists(pkl)
    vfn_tag = f"[OK]" if vfn_ok else "[MISS]"
    pkl_tag = f"[OK]" if pkl_ok else "[MISS]"
    print(f"  {vfn_tag} {vfn:<45}  {pkl_tag} {pkl}")
    if not vfn_ok or not pkl_ok:
        all_present = False
if not all_present:
    print("\nWARNING: Some files missing.")
else:
    print("\nAll 1v1 base models present.")

---
## Phase 5 — Fine-tune vs Bots (1v1 only)
Loads `model.pkl` saved by Phase 1 & 3, warm-starts each GBC/GBR, and fits additional trees on live bot-match data only (no replay mixing).

| Script | Games |
|---|---|
| classifier 1v1  | 16 enemies × 10 = 160 games |
| regressor 1v1   | 16 enemies × 10 = 160 games |

4v4 bot fine-tuning runs in a separate notebook: `train_4v4_bots_kaggle.ipynb`

In [ ]:
if RUN_VS_BOTS:
    print("=== 1v1 Classifier vs Bots ===")
    !python classifier/train_1v1_vs_bots.py --games-per-enemy 10
else:
    print("Skipped.")

In [ ]:
if RUN_VS_BOTS:
    print("=== 1v1 Regressor vs Bots ===")
    !python regressor/train_vs_bots.py --games-per-enemy 10
else:
    print("Skipped.")

---
## Output Summary
Base models (Phases 1–4) + 1v1 fine-tuned models (Phase 5) are copied to `/kaggle/working/` for download.
4v4 fine-tuned outputs come from `train_4v4_bots_kaggle.ipynb`.

In [ ]:
import shutil, os
outputs = [
    ("classifier/value_fn_1v1.py",    "classifier_value_fn_1v1.py"),
    ("regressor/value_fn_1v1.py",     "regressor_value_fn_1v1.py"),
]
print("Output files:")
print("-" * 60)
all_ok = True
for src, dest_name in outputs:
    if os.path.exists(src):
        sz  = os.path.getsize(src) / 1024
        dst = f"/kaggle/working/{dest_name}"
        shutil.copy(src, dst)
        print(f"  [OK]   {src:<42} {sz:>7.1f} KB")
    else:
        print(f"  [MISS] {src}")
        all_ok = False
print("-" * 60)
if all_ok:
    print("All 1v1 models trained. Download files from the Output panel.")
